In [1]:
import numpy as np
import pyuvdata
from calico import calibration_qa
import matplotlib.pyplot as plt
import scipy

In [2]:
cal_path = "/lustre/pipeline/calibration/results/2026-04-19/17h/successful/20260430_233320/tables/calibration_2026-04-19_17h.B.flagged"
cal = pyuvdata.UVCal()
cal.read(cal_path)

# Collapse spws
gain = np.ma.masked_where(cal.flag_array, cal.gain_array)
gain = np.ma.sum(gain, axis=2)
freq_inds = np.arange(cal.Nfreqs)
use_freqs = cal.freq_array

gain = gain[:, freq_inds, :]
gain.shape = (cal.Nants_data, len(freq_inds), 1, cal.Njones)

# problem. we cant combine flags because the time staggering means that every time is flagged at least once
flags = np.zeros(gain.shape, dtype=bool)
flags[np.abs(gain) > 0.999] = 1  # in my file, gain=1 was flagged.
flags[np.abs(gain) < 1e-9] = 1  # no gain no pain

cal.select(times=cal.time_array[-1], frequencies=use_freqs)
cal.flex_spw_id_array = np.zeros_like(use_freqs, dtype=int)
cal.spw_array = np.array([0])
cal.Nspws = 1
cal.gain_array = np.array(gain)
cal.flag_array = np.array(flags)

# Conjugate gains
cal.gain_array = np.conj(cal.gain_array)

Setting telescope_location to value in known_telescopes for OVRO-LWA.
Unknown polarization basis for solutions, jones_array values may be spurious.
Unknown x_orientation basis for solutions, assuming "east".


In [3]:
use_freqs = [13, 18, 23, 27, 32, 36, 41, 46, 50, 55, 59, 64, 69, 73, 78, 82]
for freq_ind, freq in enumerate(use_freqs):
    cal_new = pyuvdata.UVCal()
    cal_new.read(f"/lustre/rbyrne/2026-04-19/20260419_112543-112633_{freq}MHz_calico.calfits")
    if freq_ind == 0:
        cal_calico = cal_new
    else:
        cal_calico.fast_concat(cal_new, "freq", inplace=True)


In [4]:
use_freqs = [13, 18, 23, 27, 32, 36, 41, 46, 50, 55, 59, 64, 69, 73, 78, 82]
for freq_ind, freq in enumerate(use_freqs):
    cal_new = pyuvdata.UVCal()
    cal_new.read(f"/lustre/rbyrne/2026-04-19/20260419_112543-112633_{freq}MHz_calico_updated_beam.calfits")
    if freq_ind == 0:
        cal_calico_updated_beam = cal_new
    else:
        cal_calico_updated_beam.fast_concat(cal_new, "freq", inplace=True)

In [5]:
use_frequencies = cal.freq_array[np.where((28352050.78125 <= cal.freq_array) & (41295898.4375 >= cal.freq_array))]

In [6]:
cal.select(frequencies=use_frequencies, inplace=True)
#cal_calico.select(frequencies=use_frequencies, inplace=True)
#cal_calico_updated_beam.select(frequencies=use_frequencies, inplace=True)

In [7]:
#calibration_qa.plot_gains(cal, savefig=False, cal_name="pipeline", zero_mean_phase=True)

In [8]:
#calibration_qa.plot_gains(cal_calico, savefig=False, cal_name="Calico", zero_mean_phase=True)

In [11]:
calibration_qa.plot_gains(cal_calico_updated_beam, cal2=cal_calico, savefig=True, cal_name=["new_beam", "old_beam"], zero_mean_phase=True, plot_output_dir="/lustre/rbyrne/2026-04-19/gain_plots")

Mean of empty slice
Mean of empty slice
Mean of empty slice
Mean of empty slice
